# Assignment 2 – Data Loading, Storage and File Formats (Sales Data)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

## 1. Load from CSV, Excel, and JSON

In [ ]:
part1 = pd.read_csv('sales_part1.csv')
print('CSV shape:', part1.shape)
part1.head(2)

In [ ]:
part2 = pd.read_excel('sales_part2.xlsx')
print('Excel shape:', part2.shape)
part2.head(2)

In [ ]:
with open('sales_part3.json') as f:
    part3 = pd.DataFrame(json.load(f))
print('JSON shape:', part3.shape)
part3.head(2)

## 2. Explore Structure

In [ ]:
for name, df in [('CSV', part1), ('Excel', part2), ('JSON', part3)]:
    print(f'--- {name} ---')
    print(df.dtypes)
    print('Missing values:\n', df.isnull().sum()[df.isnull().sum() > 0])
    print()

## 3. Data Cleaning

In [ ]:
# Standardise column names across all parts
def clean_df(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df = df.drop_duplicates()
    df = df.dropna(subset=['order_id'])
    return df

part1 = clean_df(part1)
part2 = clean_df(part2)
part3 = clean_df(part3)

## 4. Merge into Unified DataFrame

In [ ]:
df = pd.concat([part1, part2, part3], ignore_index=True)
df = df.drop_duplicates(subset=['order_id', 'row_id'])
print('Unified shape:', df.shape)

## 5. Data Transformation

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date']  = pd.to_datetime(df['ship_date'])
df['days_to_ship'] = (df['ship_date'] - df['order_date']).dt.days
df['year'] = df['order_date'].dt.year

df[['sales', 'profit', 'quantity', 'discount']] = df[['sales', 'profit', 'quantity', 'discount']].apply(pd.to_numeric, errors='coerce')
df.dtypes

## 6. Descriptive Analysis & Visualisation

In [ ]:
print('Total Sales : ${}'.format(df['sales'].sum()))
print('Avg Order Value: ${}'.format(df['sales'].mean()))
print('Total Profit: ${}'.format(df['profit'].sum()))

print('\nSales by Category:')
print(df.groupby('category')['sales'].sum().sort_values(ascending=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sales by Category
cat_sales = df.groupby('category')['sales'].sum()
axes[0].bar(cat_sales.index, cat_sales.values, color=['steelblue', 'seagreen', 'tomato'])
axes[0].set_title('Total Sales by Category')
axes[0].set_ylabel('Sales ($)')

# Product category distribution (pie)
axes[1].pie(cat_sales.values, labels=cat_sales.index, autopct='%.1f%%')
axes[1].set_title('Category Distribution')

# Yearly sales trend
yearly = df.groupby('year')['sales'].sum()
axes[2].plot(yearly.index, yearly.values, marker='o')
axes[2].set_title('Yearly Sales Trend')
axes[2].set_ylabel('Sales ($)')
axes[2].set_xlabel('Year')

plt.tight_layout()
plt.show()

In [ ]:
# Sales by Region
region_sales = df.groupby('region')['sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(6, 4))
plt.bar(region_sales.index, region_sales.values, color='cornflowerblue')
plt.title('Sales by Region')
plt.ylabel('Sales ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Save the cleaned dataset to drive
df.to_csv("unified_sales.csv", index=False)
print('Unified dataset saved.')